# Kaggle AutoResearch Collective — Borg/Locutus 0.97122+ Refinement Notebook v3 🧪🛡️🧬

**Target:** improve a known high-score submission lock (`0.97122`) without fake/toy logic.

This notebook performs **local Kaggle autoresearch** only:

1. Scans `/kaggle/input` for real competition data and real attached submissions.
2. Validates every candidate against `sample_submission.csv`.
3. Finds the best available `0.97122` anchor when present.
4. Builds conservative **micro-flip** candidates around that anchor.
5. Generates a candidate pack for manual leaderboard testing.
6. Writes one safe final `/kaggle/working/submission.csv`.

No synthetic labels. No guessed targets. If the notebook lacks real inputs, it fails hard.


## v2 SourceFix notes

- Direct input files named `submission.csv` are now treated as valid candidate submissions instead of being skipped.
- Default guardian mode now **requires a real anchor/source submission**. If none is attached, it stops instead of writing a low-scoring model-only fallback.
- Model-only fallback is still available by setting `KAGGLE_AR_MODE=model_fallback` or `KAGGLE_AR_ALLOW_MODEL_ONLY=1`.


**v3 Borg patch:** additive collective voter layer. Preserves the anchor/guardian baseline and only proposes strict micro-flips when source consensus is strong.

## Runtime controls ⚙️

| Variable | Default | Purpose |
|---|---:|---|
| `KAGGLE_AR_MODE` | `guardian_97122` | `guardian_97122`, `anchor_lock`, `weighted_stack`, `majority_stack`, `model_fallback`, `auto` |
| `KAGGLE_AR_ANCHOR_SCORE` | `0.97122` | preferred public-score anchor parsed from filenames |
| `KAGGLE_AR_NO_TOY` | `1` | fail instead of producing fake predictions |
| `KAGGLE_AR_MAX_FLIP_RATE` | `0.003` | max rows changed vs anchor for default guardian pick |
| `KAGGLE_AR_STRICT_MARGIN` | `0.84` | weighted-vote share needed to flip anchor rows |
| `KAGGLE_AR_MIN_AGREE` | `3` | minimum source count supporting a flip |
| `KAGGLE_AR_DEDUPE` | `1` | keep one copy per identical submission hash |
| `KAGGLE_AR_USE_OPTIONAL_MODELS` | `0` | enable LightGBM/XGBoost/CatBoost if installed |
| `KAGGLE_AR_MODEL_ASSIST` | `0` | allow model fallback as a low-weight tie-break voter |
| `KAGGLE_AR_FOLDS` | `5` | CV folds for model fallback |
| `KAGGLE_INPUT_ROOT` | auto | local override for testing outside Kaggle |

Final output:

```text
/kaggle/working/submission.csv
```

Candidate pack:

```text
/kaggle/working/autoresearch/candidates/*.csv
```

| `KAGGLE_AR_REQUIRE_SOURCE` | `1` | stop guardian mode when no real source/anchor submission is attached |
| `KAGGLE_AR_ALLOW_MODEL_ONLY` | `0` | permit model-only fallback when source submissions are missing |
| `KAGGLE_AR_ANCHOR_PATH` | empty | optional explicit path to a known anchor CSV inside `/kaggle/input` |


### Borg/Locutus v3 controls 🧬

| Variable | Default | Purpose |
|---|---:|---|
| `KAGGLE_BORG_ENABLE` | `1` | Enable additive Borg collective candidate generation. |
| `KAGGLE_BORG_VOTERS` | `3,15,16,17,52` | Ranked source positions used as default Borg voters. Missing positions are skipped/filled safely. |
| `KAGGLE_BORG_MIN_AGREE` | `5` | Minimum Borg voter agreement for strict guardian flips. |
| `KAGGLE_BORG_MAX_FLIP_RATE` | inherits `KAGGLE_AR_MAX_FLIP_RATE` | Hard cap on Borg micro-flips vs anchor. |
| `KAGGLE_AR_MODE=borg_collective` | optional | Force Borg candidate selection; otherwise `guardian_97122`/`auto` can still pick Borg candidates if safest. |


In [1]:
from __future__ import annotations

import os
import re
import io
import json
import math
import time
import zipfile
import warnings
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 240)

try:
    from IPython.display import display
except Exception:
    display = print

SEED = int(os.getenv("KAGGLE_AR_SEED", "918"))
np.random.seed(SEED)

_DEFAULT_ROOT = Path("/kaggle/input") if Path("/kaggle/input").exists() else Path(".")
ROOT = Path(os.getenv("KAGGLE_INPUT_ROOT", str(_DEFAULT_ROOT))).resolve()
WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(os.getenv("KAGGLE_WORK_ROOT", ".")).resolve()
WORK.mkdir(parents=True, exist_ok=True)

AR_DIR = WORK / "autoresearch"
CAND_DIR = AR_DIR / "candidates"
AR_DIR.mkdir(parents=True, exist_ok=True)
CAND_DIR.mkdir(parents=True, exist_ok=True)

OUT = WORK / "submission.csv"
REPORT_MD = WORK / "kaggle_autoresearch_report.md"
INVENTORY_CSV = AR_DIR / "input_inventory.csv"
CORE_PROFILE_CSV = AR_DIR / "core_table_profile.csv"
VALID_SOURCES_CSV = AR_DIR / "valid_source_submissions.csv"
SOURCE_AUDIT_CSV = AR_DIR / "source_audit.csv"
PAIRWISE_CSV = AR_DIR / "source_pairwise_disagreement.csv"
CANDIDATES_CSV = AR_DIR / "candidate_manifest.csv"
OOF_CSV = AR_DIR / "model_oof_report.csv"
DEBUG_CSV = AR_DIR / "debug_predictions.csv"
FLIPS_CSV = AR_DIR / "guardian_flip_rows.csv"

CANONICAL_CLASSES = ["GALAXY", "STAR", "QSO"]
CLASS_TO_INT = {c: i for i, c in enumerate(CANONICAL_CLASSES)}
INT_TO_CLASS = {i: c for c, i in CLASS_TO_INT.items()}

MODE = os.getenv("KAGGLE_AR_MODE", "guardian_97122").strip().lower()
NO_TOY = os.getenv("KAGGLE_AR_NO_TOY", "1").strip().lower() not in {"0", "false", "no", "off"}
N_FOLDS = int(os.getenv("KAGGLE_AR_FOLDS", "5"))
ANCHOR_SCORE = float(os.getenv("KAGGLE_AR_ANCHOR_SCORE", "0.97122"))
STRICT_MARGIN = float(os.getenv("KAGGLE_AR_STRICT_MARGIN", "0.84"))
MIN_AGREE = int(os.getenv("KAGGLE_AR_MIN_AGREE", "3"))
MAX_FLIP_RATE = float(os.getenv("KAGGLE_AR_MAX_FLIP_RATE", "0.003"))
DEDUPE = os.getenv("KAGGLE_AR_DEDUPE", "1").strip().lower() not in {"0", "false", "no", "off"}
USE_OPTIONAL_MODELS = os.getenv("KAGGLE_AR_USE_OPTIONAL_MODELS", "0").strip().lower() in {"1", "true", "yes", "on"}
MODEL_ASSIST = os.getenv("KAGGLE_AR_MODEL_ASSIST", "0").strip().lower() in {"1", "true", "yes", "on"}
REQUIRE_SOURCE = os.getenv("KAGGLE_AR_REQUIRE_SOURCE", "1").strip().lower() not in {"0", "false", "no", "off"}
ALLOW_MODEL_ONLY = os.getenv("KAGGLE_AR_ALLOW_MODEL_ONLY", "0").strip().lower() in {"1", "true", "yes", "on"}
ANCHOR_PATH_HINT = os.getenv("KAGGLE_AR_ANCHOR_PATH", "").strip()
SOURCE_TEMP_DEFAULT = float(os.getenv("KAGGLE_AR_SOURCE_TEMP", "6500"))

print("ROOT:", ROOT)
print("WORK:", WORK)
print("MODE:", MODE)
print("ANCHOR_SCORE:", ANCHOR_SCORE)
print("STRICT_MARGIN:", STRICT_MARGIN)
print("MIN_AGREE:", MIN_AGREE)
print("MAX_FLIP_RATE:", MAX_FLIP_RATE)
print("DEDUPE:", DEDUPE)
print("USE_OPTIONAL_MODELS:", USE_OPTIONAL_MODELS)
print("MODEL_ASSIST:", MODEL_ASSIST)
print("REQUIRE_SOURCE:", REQUIRE_SOURCE)
print("ALLOW_MODEL_ONLY:", ALLOW_MODEL_ONLY)
print("ANCHOR_PATH_HINT:", ANCHOR_PATH_HINT or "<auto>")


ROOT: /kaggle/input
WORK: /kaggle/working
MODE: guardian_97122
ANCHOR_SCORE: 0.97122
STRICT_MARGIN: 0.84
MIN_AGREE: 3
MAX_FLIP_RATE: 0.003
DEDUPE: True
USE_OPTIONAL_MODELS: False
MODEL_ASSIST: False
REQUIRE_SOURCE: True
ALLOW_MODEL_ONLY: False
ANCHOR_PATH_HINT: <auto>


In [2]:
def write_report(title: str, lines: list[str] | None = None, *, reset: bool = False) -> None:
    mode = "w" if reset else "a"
    with open(REPORT_MD, mode, encoding="utf-8") as f:
        if reset:
            f.write("# Kaggle AutoResearch 0.97122 Refinement Report\n")
            f.write(f"\nGenerated: {time.strftime('%Y-%m-%d %H:%M:%S')}\n")
            f.write(f"\nROOT: `{ROOT}`\n\nWORK: `{WORK}`\n")
        f.write(f"\n\n## {title}\n\n")
        for line in lines or []:
            f.write(str(line).rstrip() + "\n")


def safe_rel(path: Path) -> str:
    try:
        return str(path.resolve().relative_to(ROOT.resolve()))
    except Exception:
        return str(path)


def fast_line_count(path: Path) -> int | None:
    try:
        with open(path, "rb") as f:
            return max(sum(1 for _ in f) - 1, 0)
    except Exception:
        return None


def file_score_from_name(name: str) -> float:
    """Extract public score-like value from filenames such as `submission_0.97122.csv`."""
    candidates = re.findall(r"(?<!\d)(0\.\d{4,8})(?!\d)", str(name))
    if not candidates:
        return float("nan")
    return max(float(x) for x in candidates)


def canonicalize_label(x):
    if pd.isna(x):
        return x
    y = str(x).strip().upper()
    aliases = {
        "GALAXY": "GALAXY", "GALAXIA": "GALAXY", "GALAXIE": "GALAXY", "GALASSIA": "GALAXY", "GALÁXIA": "GALAXY",
        "STAR": "STAR", "ESTRELLA": "STAR", "ÉTOILE": "STAR", "ETOILE": "STAR", "STERN": "STAR", "STELLA": "STAR", "ESTRELA": "STAR",
        "QSO": "QSO", "QUASAR": "QSO", "CUASAR": "QSO", "CUÁSAR": "QSO",
    }
    return aliases.get(y, y)


def validate_id_class(df: pd.DataFrame, sample: pd.DataFrame, *, name: str) -> pd.DataFrame:
    cols_lower = {str(c).lower(): c for c in df.columns}
    if "id" not in cols_lower or "class" not in cols_lower:
        raise ValueError(f"{name}: missing required id/class columns; columns={list(df.columns)}")
    out = df[[cols_lower["id"], cols_lower["class"]]].copy()
    out.columns = ["id", "class"]
    out["class"] = out["class"].map(canonicalize_label)
    if len(out) != len(sample):
        raise ValueError(f"{name}: row count {len(out)} != sample row count {len(sample)}")
    if out["id"].isna().any() or out["class"].isna().any():
        raise ValueError(f"{name}: null id/class values detected")
    if not out["id"].equals(sample["id"]):
        if set(out["id"].tolist()) == set(sample["id"].tolist()):
            out = sample[["id"]].merge(out, on="id", how="left")
        else:
            raise ValueError(f"{name}: id values do not match sample_submission.csv")
    bad = sorted(set(out["class"].unique()) - set(CANONICAL_CLASSES))
    if bad:
        raise ValueError(f"{name}: illegal labels {bad}; allowed={CANONICAL_CLASSES}")
    return out[["id", "class"]]


def dataframe_hash(df: pd.DataFrame) -> str:
    import hashlib
    h = hashlib.sha256()
    h.update(pd.util.hash_pandas_object(df, index=False).values.tobytes())
    return h.hexdigest()[:16]


def labels_to_int(labels: pd.Series | np.ndarray) -> np.ndarray:
    return pd.Series(labels).map(CLASS_TO_INT).astype(np.int16).to_numpy()


def ints_to_labels(arr: np.ndarray) -> pd.Series:
    return pd.Series(arr).map(INT_TO_CLASS)


def make_submission(pred: pd.Series | np.ndarray, sample: pd.DataFrame) -> pd.DataFrame:
    out = sample[["id"]].copy()
    out["class"] = pd.Series(pred).map(canonicalize_label).values
    return validate_id_class(out, sample, name="candidate")

write_report("Start", [
    f"Mode: `{MODE}`",
    f"Anchor score hint: `{ANCHOR_SCORE}`",
    f"No-toy mode: `{NO_TOY}`",
    f"Require source: `{REQUIRE_SOURCE}`",
    f"Allow model only: `{ALLOW_MODEL_ONLY}`",
    f"Anchor path hint: `{ANCHOR_PATH_HINT or '<auto>'}`",
    f"Seed: `{SEED}`",
], reset=True)


## 1. AutoResearch file inventory 🔎


In [3]:
def inventory_inputs(root: Path = ROOT, max_files: int = 75000) -> pd.DataFrame:
    rows = []
    files = [p for p in root.rglob("*") if p.is_file()] if root.exists() else []
    for p in files[:max_files]:
        row = {
            "path": str(p),
            "relpath": safe_rel(p),
            "name": p.name,
            "suffix": p.suffix.lower(),
            "size_bytes": int(p.stat().st_size),
            "rows_estimate": None,
            "n_columns": None,
            "columns": "",
            "read_error": "",
        }
        if p.suffix.lower() == ".csv":
            try:
                head = pd.read_csv(p, nrows=5)
                row["n_columns"] = int(len(head.columns))
                row["columns"] = "|".join(map(str, head.columns.tolist()))
                row["rows_estimate"] = fast_line_count(p)
            except Exception as e:
                row["read_error"] = repr(e)[:500]
        elif p.suffix.lower() == ".zip":
            try:
                with zipfile.ZipFile(p) as z:
                    row["columns"] = "zip:" + "|".join(z.namelist()[:20])
            except Exception as e:
                row["read_error"] = repr(e)[:500]
        rows.append(row)
    inv = pd.DataFrame(rows)
    if len(inv):
        inv = inv.sort_values(["suffix", "name", "size_bytes"], ascending=[True, True, False]).reset_index(drop=True)
    inv.to_csv(INVENTORY_CSV, index=False)
    write_report("Input inventory", [f"Files found: `{len(inv)}`", f"Saved: `{INVENTORY_CSV}`"])
    return inv

inventory = inventory_inputs()
display(inventory.head(40))


,path,relpath,name,suffix,size_bytes,rows_estimate,n_columns,columns,read_error
0,/kaggle/input/datasets/nina2025/ps-s6e6/0.9665...,datasets/nina2025/ps-s6e6/0.96652.csv,0.96652.csv,.csv,3230612,247435.0,2.0,id|class,
1,/kaggle/input/datasets/nina2025/ps-s6e6/0.9670...,datasets/nina2025/ps-s6e6/0.96707.csv,0.96707.csv,.csv,3230811,247435.0,2.0,id|class,
2,/kaggle/input/datasets/nina2025/ps-s6e6/0.9672...,datasets/nina2025/ps-s6e6/0.96722.csv,0.96722.csv,.csv,3231753,247435.0,2.0,id|class,
3,/kaggle/input/datasets/nina2025/ps-s6e6/0.9675...,datasets/nina2025/ps-s6e6/0.96751.csv,0.96751.csv,.csv,3230396,247435.0,2.0,id|class,
4,/kaggle/input/datasets/nina2025/ps-s6e6/0.9675...,datasets/nina2025/ps-s6e6/0.96755.csv,0.96755.csv,.csv,3230823,247435.0,2.0,id|class,
5,/kaggle/input/datasets/nina2025/ps-s6e6/0.9676...,datasets/nina2025/ps-s6e6/0.96766.csv,0.96766.csv,.csv,3230971,247435.0,2.0,id|class,
6,/kaggle/input/datasets/nina2025/ps-s6e6/0.9677...,datasets/nina2025/ps-s6e6/0.96771.b.csv,0.96771.b.csv,.csv,3231224,247435.0,2.0,id|class,
7,/kaggle/input/datasets/nina2025/ps-s6e6/0.9677...,datasets/nina2025/ps-s6e6/0.96771.csv,0.96771.csv,.csv,3231776,247435.0,2.0,id|class,
8,/kaggle/input/datasets/nina2025/ps-s6e6/0.9680...,datasets/nina2025/ps-s6e6/0.96807.csv,0.96807.csv,.csv,3230401,247435.0,2.0,id|class,
9,/kaggle/input/datasets/nina2025/ps-s6e6/0.9688...,datasets/nina2025/ps-s6e6/0.96889.csv,0.96889.csv,.csv,3232320,247435.0,2.0,id|class,


## 2. Locate core Kaggle tables 📁


In [4]:
def find_by_name(filename: str, *, required: bool = True) -> Path | None:
    hits = list(ROOT.rglob(filename)) if ROOT.exists() else []
    if not hits:
        if required:
            raise FileNotFoundError(f"Missing {filename!r} under {ROOT}. Attach the competition data.")
        return None
    def rank(p: Path):
        s = str(p).lower()
        return (
            0 if "playground-series-s6e6" in s else 1,
            0 if "competition" in s or "competitions" in s else 1,
            len(str(p)),
        )
    return sorted(hits, key=rank)[0]

sample_path = find_by_name("sample_submission.csv", required=True)
test_path = find_by_name("test.csv", required=False)
train_path = find_by_name("train.csv", required=False)

sample = pd.read_csv(sample_path)
if "id" not in sample.columns or "class" not in sample.columns:
    raise ValueError(f"sample_submission.csv must contain id,class. Found {sample.columns.tolist()}")
sample = sample[["id", "class"]].copy()

core_rows = []
for label, path in [("sample", sample_path), ("train", train_path), ("test", test_path)]:
    if path is None:
        core_rows.append({"table": label, "path": None, "rows": None, "columns": None, "id_unique": None})
        continue
    df_head = pd.read_csv(path, nrows=5)
    rows = fast_line_count(path)
    id_unique = None
    if "id" in df_head.columns:
        try:
            id_unique = pd.read_csv(path, usecols=["id"])["id"].is_unique
        except Exception:
            id_unique = None
    core_rows.append({"table": label, "path": str(path), "rows": rows, "columns": "|".join(map(str, df_head.columns)), "id_unique": id_unique})

core_profile = pd.DataFrame(core_rows)
core_profile.to_csv(CORE_PROFILE_CSV, index=False)
write_report("Core tables", [f"sample: `{sample_path}`", f"train: `{train_path}`", f"test: `{test_path}`", f"sample rows: `{len(sample)}`"])
display(core_profile)
print("Sample rows:", len(sample))


,table,path,rows,columns,id_unique
0,sample,/kaggle/input/competitions/playground-series-s...,247435,id|class,True
1,train,/kaggle/input/competitions/playground-series-s...,577347,id|alpha|delta|u|g|r|i|z|redshift|spectral_typ...,True
2,test,/kaggle/input/competitions/playground-series-s...,247435,id|alpha|delta|u|g|r|i|z|redshift|spectral_typ...,True


Sample rows: 247435


## 3. Discover real source submissions 🧬

Keeps only files that match the actual sample IDs and legal labels. Supports direct `.csv` files and `.csv` files inside `.zip` archives.


In [5]:
CORE_DIRECT_NAMES = {"train.csv", "test.csv", "sample_submission.csv"}
CORE_RESOLVED_PATHS = {p.resolve() for p in [sample_path, train_path, test_path] if p is not None}


def is_core_direct_csv(path: Path) -> bool:
    """Skip real competition tables, but do NOT skip user-added `submission.csv` anchors."""
    try:
        if path.resolve() in CORE_RESOLVED_PATHS:
            return True
    except Exception:
        pass
    lower = path.name.lower()
    # Important v2 fix: `submission.csv` is NOT a core file. Many Kaggle datasets store prior LB submissions under that exact name.
    return lower in CORE_DIRECT_NAMES


def iter_csv_sources(root: Path):
    # Explicit anchor path first, if provided.
    if ANCHOR_PATH_HINT:
        hint = Path(ANCHOR_PATH_HINT)
        if not hint.is_absolute():
            hint = root / hint
        if hint.exists() and hint.suffix.lower() == ".csv":
            yield {"kind": "csv", "path": hint, "member": "", "display_name": f"EXPLICIT_ANCHOR::{hint.name}", "score_name": f"anchor_{ANCHOR_SCORE}_{hint.name}"}

    for p in sorted(root.rglob("*.csv")) if root.exists() else []:
        if is_core_direct_csv(p):
            continue
        yield {"kind": "csv", "path": p, "member": "", "display_name": p.name, "score_name": p.name}

    for p in sorted(root.rglob("*.zip")) if root.exists() else []:
        try:
            with zipfile.ZipFile(p) as z:
                for member in sorted(z.namelist()):
                    if member.lower().endswith(".csv") and Path(member).name.lower() not in CORE_DIRECT_NAMES:
                        yield {"kind": "zip_csv", "path": p, "member": member, "display_name": f"{p.name}::{member}", "score_name": f"{p.name}_{Path(member).name}"}
        except Exception:
            continue


def read_source_head(ref: dict, nrows: int = 5) -> pd.DataFrame:
    if ref["kind"] == "csv":
        return pd.read_csv(ref["path"], nrows=nrows)
    with zipfile.ZipFile(ref["path"]) as z:
        with z.open(ref["member"]) as f:
            return pd.read_csv(f, nrows=nrows)


def read_source_full(ref: dict) -> pd.DataFrame:
    if ref["kind"] == "csv":
        return pd.read_csv(ref["path"])
    with zipfile.ZipFile(ref["path"]) as z:
        with z.open(ref["member"]) as f:
            return pd.read_csv(f)


def discover_source_submissions() -> tuple[pd.DataFrame, dict[str, pd.DataFrame]]:
    meta = []
    frames: dict[str, pd.DataFrame] = {}
    seen_hashes: dict[str, str] = {}
    checked = 0
    rejected = []

    for ref in iter_csv_sources(ROOT):
        checked += 1
        try:
            head = read_source_head(ref, nrows=3)
            head_cols = {str(c).lower() for c in head.columns}
            if not {"id", "class"}.issubset(head_cols):
                rejected.append({"name": ref["display_name"], "reason": "missing id/class", "columns": "|".join(map(str, head.columns))})
                continue
            raw = read_source_full(ref)
            sub = validate_id_class(raw, sample, name=ref["display_name"])
            h = dataframe_hash(sub)
            duplicate_of = seen_hashes.get(h)
            if duplicate_of is None:
                seen_hashes[h] = ref["display_name"]
            key = f"src_{len(meta):03d}_{re.sub('[^A-Za-z0-9_.-]+', '_', ref['display_name'])[:120]}"
            score = file_score_from_name(ref["score_name"])
            frames[key] = sub
            meta.append({
                "key": key,
                "kind": ref["kind"],
                "path": str(ref["path"]),
                "relpath": safe_rel(ref["path"]),
                "member": ref["member"],
                "filename": ref["display_name"],
                "rows": len(sub),
                "hash": h,
                "duplicate_of": duplicate_of or "",
                "filename_score": score,
                "score_distance_to_anchor": abs(score - ANCHOR_SCORE) if not math.isnan(score) else np.nan,
                "GALAXY": int((sub["class"] == "GALAXY").sum()),
                "STAR": int((sub["class"] == "STAR").sum()),
                "QSO": int((sub["class"] == "QSO").sum()),
            })
        except Exception as e:
            rejected.append({"name": ref.get("display_name", "?"), "reason": repr(e)[:300], "columns": ""})
            continue

    columns = ["key", "kind", "path", "relpath", "member", "filename", "rows", "hash", "duplicate_of", "filename_score", "score_distance_to_anchor", "GALAXY", "STAR", "QSO"]
    source_meta = pd.DataFrame(meta, columns=columns)
    if len(source_meta):
        source_meta = source_meta.sort_values(
            ["filename_score", "score_distance_to_anchor", "filename", "path"],
            ascending=[False, True, True, True],
            na_position="last",
        ).reset_index(drop=True)
    source_meta.to_csv(VALID_SOURCES_CSV, index=False)
    pd.DataFrame(rejected).to_csv(AR_DIR / "rejected_source_candidates.csv", index=False)
    print("Checked source-like CSVs:", checked)
    print("Rejected source-like CSVs:", len(rejected))
    return source_meta, frames

source_meta, source_frames = discover_source_submissions()
write_report("Valid source submissions", [
    f"Valid source files: `{len(source_frames)}`",
    f"Saved: `{VALID_SOURCES_CSV}`",
    f"Rejected candidates: `{AR_DIR / 'rejected_source_candidates.csv'}`",
])
print("Valid source submissions:", len(source_frames))
if not len(source_frames):
    print("⚠️ No valid source submissions found. Attach your 0.97122 CSV as a Kaggle Dataset. A file named submission.csv is now accepted in v2.")
display(source_meta.head(60))


Checked source-like CSVs: 62
Rejected source-like CSVs: 3
Valid source submissions: 59


,key,kind,path,relpath,member,filename,rows,hash,duplicate_of,filename_score,score_distance_to_anchor,GALAXY,STAR,QSO
0,src_058_0.97122.csv,csv,/kaggle/input/datasets/nina2025/ps-s6e6/0.9712...,datasets/nina2025/ps-s6e6/0.97122.csv,,0.97122.csv,247435,4fa2346b2a213bd4,,0.97122,0.00000,156886,39067,51482
1,src_057_0.97114.csv,csv,/kaggle/input/datasets/nina2025/ps-s6e6/0.9711...,datasets/nina2025/ps-s6e6/0.97114.csv,,0.97114.csv,247435,9a7259a4c3c02c74,,0.97114,0.00008,156816,39181,51438
2,src_055_0.97111.b.csv,csv,/kaggle/input/datasets/nina2025/ps-s6e6/0.9711...,datasets/nina2025/ps-s6e6/0.97111.b.csv,,0.97111.b.csv,247435,35f18b9fb3712a63,,0.97111,0.00011,157000,38987,51448
3,src_056_0.97111.csv,csv,/kaggle/input/datasets/nina2025/ps-s6e6/0.9711...,datasets/nina2025/ps-s6e6/0.97111.csv,,0.97111.csv,247435,35f18b9fb3712a63,0.97111.b.csv,0.97111,0.00011,157000,38987,51448
4,src_054_0.97108.csv,csv,/kaggle/input/datasets/nina2025/ps-s6e6/0.9710...,datasets/nina2025/ps-s6e6/0.97108.csv,,0.97108.csv,247435,fab7f937d1c4b945,,0.97108,0.00014,157082,38931,51422
5,src_053_0.97106.csv,csv,/kaggle/input/datasets/nina2025/ps-s6e6/0.9710...,datasets/nina2025/ps-s6e6/0.97106.csv,,0.97106.csv,247435,eb2f4735a1866ed8,,0.97106,0.00016,156882,39050,51503
6,src_052_0.97102.csv,csv,/kaggle/input/datasets/nina2025/ps-s6e6/0.9710...,datasets/nina2025/ps-s6e6/0.97102.csv,,0.97102.csv,247435,aaaa081a9dc62174,,0.97102,0.00020,156990,38996,51449
7,src_051_0.97101.csv,csv,/kaggle/input/datasets/nina2025/ps-s6e6/0.9710...,datasets/nina2025/ps-s6e6/0.97101.csv,,0.97101.csv,247435,55d93c7bb92a5dbe,,0.97101,0.00021,156993,38990,51452
8,src_050_0.97093.csv,csv,/kaggle/input/datasets/nina2025/ps-s6e6/0.9709...,datasets/nina2025/ps-s6e6/0.97093.csv,,0.97093.csv,247435,f465e80c87ada572,,0.97093,0.00029,157047,38946,51442
9,src_049_0.97092.csv,csv,/kaggle/input/datasets/nina2025/ps-s6e6/0.9709...,datasets/nina2025/ps-s6e6/0.97092.csv,,0.97092.csv,247435,18afd16e44c55389,,0.97092,0.00030,157111,38891,51433


## 4. Source audit, anchor selection, and weights 🧮


In [6]:
def source_matrix(keys: list[str]) -> np.ndarray:
    return np.vstack([labels_to_int(source_frames[k]["class"]) for k in keys])


def add_source_quality_columns(source_meta: pd.DataFrame) -> pd.DataFrame:
    meta = source_meta.copy()
    if not len(meta):
        return meta
    keys = meta["key"].tolist()
    mat = source_matrix(keys)
    n = len(keys)
    centrality = []
    for i in range(n):
        if n == 1:
            centrality.append(1.0)
        else:
            diffs = [(mat[i] == mat[j]).mean() for j in range(n) if i != j]
            centrality.append(float(np.mean(diffs)))
    meta["agreement_centrality"] = centrality
    # Score fill: use parsed filename score if present, otherwise centrality-centered pseudo-rank.
    if meta["filename_score"].notna().any():
        min_score = float(meta["filename_score"].dropna().min())
        meta["score_fill"] = meta["filename_score"].fillna(min_score - 1e-5)
    else:
        meta["score_fill"] = 0.97000 + 0.001 * (meta["agreement_centrality"] - meta["agreement_centrality"].min())
    meta["duplicate_count"] = meta.groupby("hash")["hash"].transform("count")
    meta["is_duplicate"] = meta["duplicate_of"].astype(str).ne("")
    return meta

source_meta = add_source_quality_columns(source_meta)
source_meta.to_csv(VALID_SOURCES_CSV, index=False)


def select_anchor_key(meta: pd.DataFrame) -> str | None:
    if not len(meta):
        return None
    tmp = meta.copy()
    exact = tmp[tmp["filename_score"].notna() & (np.abs(tmp["filename_score"] - ANCHOR_SCORE) <= 5e-7)]
    if len(exact):
        exact = exact.sort_values(["is_duplicate", "agreement_centrality", "filename"], ascending=[True, False, True])
        return str(exact.iloc[0]["key"])
    scored = tmp[tmp["filename_score"].notna()].copy()
    if len(scored):
        scored["dist"] = (scored["filename_score"] - ANCHOR_SCORE).abs()
        # Prefer the best score, but if something is near the stated anchor, keep it close.
        scored = scored.sort_values(["dist", "filename_score", "is_duplicate", "agreement_centrality"], ascending=[True, False, True, False])
        return str(scored.iloc[0]["key"])
    tmp = tmp.sort_values(["agreement_centrality", "is_duplicate", "filename"], ascending=[False, True, True])
    return str(tmp.iloc[0]["key"])


def ranked_source_keys(meta: pd.DataFrame, *, dedupe: bool = DEDUPE) -> list[str]:
    if not len(meta):
        return []
    tmp = meta.copy()
    tmp = tmp.sort_values(["score_fill", "agreement_centrality", "is_duplicate", "filename"], ascending=[False, False, True, True]).reset_index(drop=True)
    if dedupe:
        tmp = tmp.drop_duplicates("hash", keep="first")
    return tmp["key"].tolist()

anchor_key = select_anchor_key(source_meta)
ranked_keys = ranked_source_keys(source_meta, dedupe=DEDUPE)
if anchor_key and anchor_key not in ranked_keys:
    ranked_keys = [anchor_key] + ranked_keys

print("Anchor key:", anchor_key)
if anchor_key:
    display(source_meta[source_meta["key"] == anchor_key])
print("Ranked unique source count:", len(ranked_keys))
print("Top ranked keys:", ranked_keys[:15])
write_report("Anchor and ranking", [f"Anchor key: `{anchor_key}`", f"Ranked source count: `{len(ranked_keys)}`"])


Anchor key: src_058_0.97122.csv


,key,kind,path,relpath,member,filename,rows,hash,duplicate_of,filename_score,score_distance_to_anchor,GALAXY,STAR,QSO,agreement_centrality,score_fill,duplicate_count,is_duplicate
0,src_058_0.97122.csv,csv,/kaggle/input/datasets/nina2025/ps-s6e6/0.9712...,datasets/nina2025/ps-s6e6/0.97122.csv,,0.97122.csv,247435,4fa2346b2a213bd4,,0.97122,0.0,156886,39067,51482,0.994445,0.97122,1,False


Ranked unique source count: 56
Top ranked keys: ['src_058_0.97122.csv', 'src_057_0.97114.csv', 'src_055_0.97111.b.csv', 'src_054_0.97108.csv', 'src_053_0.97106.csv', 'src_052_0.97102.csv', 'src_051_0.97101.csv', 'src_050_0.97093.csv', 'src_049_0.97092.csv', 'src_048_0.97082.csv', 'src_047_0.97080.csv', 'src_045_0.97079.b.csv', 'src_044_0.97076.csv', 'src_042_0.97076.b.csv', 'src_043_0.97076.c.csv']


In [7]:
def compute_source_weights(keys: list[str], *, temperature: float = SOURCE_TEMP_DEFAULT) -> np.ndarray:
    if not keys:
        return np.array([], dtype=float)
    meta = source_meta.set_index("key")
    scores = []
    central = []
    dup_counts = []
    for k in keys:
        row = meta.loc[k]
        scores.append(float(row.get("score_fill", 0.97000)))
        central.append(float(row.get("agreement_centrality", 1.0)))
        dup_counts.append(float(row.get("duplicate_count", 1.0)))
    scores = np.asarray(scores, dtype=float)
    central = np.asarray(central, dtype=float)
    dup_counts = np.asarray(dup_counts, dtype=float)
    # Exponential score emphasis but clipped to avoid one filename fully dominating.
    centered = scores - np.nanmax(scores)
    score_w = np.exp(np.clip(centered * temperature, -6, 0))
    central_w = np.clip(central, 0.90, 1.00) ** 6
    dup_w = 1.0 / np.sqrt(np.maximum(dup_counts, 1.0))
    weights = score_w * central_w * dup_w
    if anchor_key in keys:
        # Anchor is the known high-LB lock; preserve it unless consensus is strong.
        weights[keys.index(anchor_key)] *= 1.15
    weights = weights / max(weights.mean(), 1e-12)
    return weights


def weighted_vote_from_sources(keys: list[str], *, temperature: float = SOURCE_TEMP_DEFAULT) -> tuple[pd.Series, pd.DataFrame]:
    if not keys:
        raise ValueError("weighted_vote_from_sources received no keys")
    mat = source_matrix(keys)
    weights = compute_source_weights(keys, temperature=temperature)
    votes = np.zeros((mat.shape[1], len(CANONICAL_CLASSES)), dtype=np.float64)
    counts = np.zeros_like(votes)
    for row, w in zip(mat, weights):
        votes[np.arange(mat.shape[1]), row] += w
        counts[np.arange(mat.shape[1]), row] += 1
    pred_i = votes.argmax(axis=1)
    vote_total = votes.sum(axis=1)
    sorted_votes = np.sort(votes, axis=1)
    top = sorted_votes[:, -1]
    second = sorted_votes[:, -2] if votes.shape[1] > 1 else np.zeros_like(top)
    top_count = counts[np.arange(mat.shape[1]), pred_i]
    conf = pd.DataFrame({
        "id": sample["id"].values,
        "votes_GALAXY": votes[:, CLASS_TO_INT["GALAXY"]],
        "votes_STAR": votes[:, CLASS_TO_INT["STAR"]],
        "votes_QSO": votes[:, CLASS_TO_INT["QSO"]],
        "count_GALAXY": counts[:, CLASS_TO_INT["GALAXY"]],
        "count_STAR": counts[:, CLASS_TO_INT["STAR"]],
        "count_QSO": counts[:, CLASS_TO_INT["QSO"]],
        "vote_total": vote_total,
        "vote_share": np.divide(top, vote_total, out=np.zeros_like(top), where=vote_total > 0),
        "vote_margin": np.divide(top - second, vote_total, out=np.zeros_like(top), where=vote_total > 0),
        "agree_count": top_count,
        "weighted_class": ints_to_labels(pred_i).values,
    })
    return ints_to_labels(pred_i), conf


def majority_vote_from_sources(keys: list[str]) -> pd.Series:
    mat = source_matrix(keys)
    preds = []
    for j in range(mat.shape[1]):
        cnt = Counter(mat[:, j])
        max_votes = max(cnt.values())
        winners = {k for k, v in cnt.items() if v == max_votes}
        # Tie-break by highest-ranked source order.
        for val in mat[:, j]:
            if val in winners:
                preds.append(val)
                break
    return ints_to_labels(np.asarray(preds, dtype=np.int16))


def audit_sources() -> pd.DataFrame:
    if not ranked_keys:
        return pd.DataFrame()
    base_key = anchor_key or ranked_keys[0]
    base = source_frames[base_key]["class"].values
    meta_idx = source_meta.set_index("key")
    rows = []
    for k in ranked_keys:
        pred = source_frames[k]["class"].values
        row = meta_idx.loc[k].to_dict() if k in meta_idx.index else {}
        row.update({
            "key": k,
            "diff_vs_anchor_rows": int((pred != base).sum()),
            "diff_vs_anchor_rate": float(np.mean(pred != base)),
        })
        rows.append(row)
    audit = pd.DataFrame(rows)
    audit.to_csv(SOURCE_AUDIT_CSV, index=False)
    return audit

source_audit = audit_sources()
display(source_audit.head(30))

pair_rows = []
pair_keys = ranked_keys[:min(25, len(ranked_keys))]
if len(pair_keys) >= 2:
    for i, a in enumerate(pair_keys):
        va = source_frames[a]["class"].values
        for b in pair_keys[i+1:]:
            vb = source_frames[b]["class"].values
            pair_rows.append({"a": a, "b": b, "diff_rows": int((va != vb).sum()), "diff_rate": float(np.mean(va != vb))})
pairwise = pd.DataFrame(pair_rows)
pairwise.to_csv(PAIRWISE_CSV, index=False)
display(pairwise.head(20))


,kind,path,relpath,member,filename,rows,hash,duplicate_of,filename_score,score_distance_to_anchor,GALAXY,STAR,QSO,agreement_centrality,score_fill,duplicate_count,is_duplicate,key,diff_vs_anchor_rows,diff_vs_anchor_rate
0,csv,/kaggle/input/datasets/nina2025/ps-s6e6/0.9712...,datasets/nina2025/ps-s6e6/0.97122.csv,,0.97122.csv,247435,4fa2346b2a213bd4,,0.97122,0.00000,156886,39067,51482,0.994445,0.97122,1,False,src_058_0.97122.csv,0,0.000000
1,csv,/kaggle/input/datasets/nina2025/ps-s6e6/0.9711...,datasets/nina2025/ps-s6e6/0.97114.csv,,0.97114.csv,247435,9a7259a4c3c02c74,,0.97114,0.00008,156816,39181,51438,0.994191,0.97114,1,False,src_057_0.97114.csv,165,0.000667
2,csv,/kaggle/input/datasets/nina2025/ps-s6e6/0.9711...,datasets/nina2025/ps-s6e6/0.97111.b.csv,,0.97111.b.csv,247435,35f18b9fb3712a63,,0.97111,0.00011,157000,38987,51448,0.994373,0.97111,2,False,src_055_0.97111.b.csv,376,0.001520
3,csv,/kaggle/input/datasets/nina2025/ps-s6e6/0.9710...,datasets/nina2025/ps-s6e6/0.97108.csv,,0.97108.csv,247435,fab7f937d1c4b945,,0.97108,0.00014,157082,38931,51422,0.994957,0.97108,1,False,src_054_0.97108.csv,588,0.002376
4,csv,/kaggle/input/datasets/nina2025/ps-s6e6/0.9710...,datasets/nina2025/ps-s6e6/0.97106.csv,,0.97106.csv,247435,eb2f4735a1866ed8,,0.97106,0.00016,156882,39050,51503,0.994227,0.97106,1,False,src_053_0.97106.csv,112,0.000453
5,csv,/kaggle/input/datasets/nina2025/ps-s6e6/0.9710...,datasets/nina2025/ps-s6e6/0.97102.csv,,0.97102.csv,247435,aaaa081a9dc62174,,0.97102,0.00020,156990,38996,51449,0.994026,0.97102,1,False,src_052_0.97102.csv,332,0.001342
6,csv,/kaggle/input/datasets/nina2025/ps-s6e6/0.9710...,datasets/nina2025/ps-s6e6/0.97101.csv,,0.97101.csv,247435,55d93c7bb92a5dbe,,0.97101,0.00021,156993,38990,51452,0.994158,0.97101,1,False,src_051_0.97101.csv,441,0.001782
7,csv,/kaggle/input/datasets/nina2025/ps-s6e6/0.9709...,datasets/nina2025/ps-s6e6/0.97093.csv,,0.97093.csv,247435,f465e80c87ada572,,0.97093,0.00029,157047,38946,51442,0.994205,0.97093,1,False,src_050_0.97093.csv,316,0.001277
8,csv,/kaggle/input/datasets/nina2025/ps-s6e6/0.9709...,datasets/nina2025/ps-s6e6/0.97092.csv,,0.97092.csv,247435,18afd16e44c55389,,0.97092,0.00030,157111,38891,51433,0.994949,0.97092,1,False,src_049_0.97092.csv,531,0.002146
9,csv,/kaggle/input/datasets/nina2025/ps-s6e6/0.9708...,datasets/nina2025/ps-s6e6/0.97082.csv,,0.97082.csv,247435,8276d7aa2d43aa80,,0.97082,0.00040,157006,38993,51436,0.994893,0.97082,1,False,src_048_0.97082.csv,715,0.002890


,a,b,diff_rows,diff_rate
0,src_058_0.97122.csv,src_057_0.97114.csv,165,0.000667
1,src_058_0.97122.csv,src_055_0.97111.b.csv,376,0.001520
2,src_058_0.97122.csv,src_054_0.97108.csv,588,0.002376
3,src_058_0.97122.csv,src_053_0.97106.csv,112,0.000453
4,src_058_0.97122.csv,src_052_0.97102.csv,332,0.001342
5,src_058_0.97122.csv,src_051_0.97101.csv,441,0.001782
6,src_058_0.97122.csv,src_050_0.97093.csv,316,0.001277
7,src_058_0.97122.csv,src_049_0.97092.csv,531,0.002146
8,src_058_0.97122.csv,src_048_0.97082.csv,715,0.002890
9,src_058_0.97122.csv,src_047_0.97080.csv,540,0.002182


## 5. Model fallback and optional fine-tuned tabular assist 🤖

Used when there are no valid source submissions, when `KAGGLE_AR_MODE=model_fallback`, or when `KAGGLE_AR_MODEL_ASSIST=1`.

Includes real feature engineering and OOF class-bias tuning for balanced accuracy. It does **not** create fake labels.


In [8]:
def infer_target_column(train: pd.DataFrame) -> str:
    for col in ["class", "target", "label", "Class", "TARGET"]:
        if col in train.columns:
            return col
    raise ValueError(f"Could not infer target column from train columns: {train.columns.tolist()}")


def add_stellar_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    # SDSS-like magnitude color indices. Only created when columns exist.
    bands = [c for c in ["u", "g", "r", "i", "z"] if c in out.columns]
    for a, b in zip(bands, bands[1:]):
        out[f"color_{a}_{b}"] = out[a] - out[b]
    for i, a in enumerate(bands):
        for b in bands[i+1:]:
            out[f"diff_{a}_{b}"] = out[a] - out[b]
            out[f"ratio_{a}_{b}"] = out[a] / (out[b].replace(0, np.nan))
    if bands:
        vals = out[bands].replace([np.inf, -np.inf], np.nan)
        out["mag_mean"] = vals.mean(axis=1)
        out["mag_std"] = vals.std(axis=1)
        out["mag_min"] = vals.min(axis=1)
        out["mag_max"] = vals.max(axis=1)
        out["mag_range"] = out["mag_max"] - out["mag_min"]
    if "redshift" in out.columns:
        rz = out["redshift"].astype(float)
        out["redshift_abs"] = rz.abs()
        out["redshift_log1p_abs"] = np.log1p(rz.abs())
        out["redshift_sq"] = rz * rz
        if "g" in out.columns and "r" in out.columns:
            out["redshift_x_g_r"] = rz * (out["g"] - out["r"])
    # Sky coordinate cyclic features.
    for col, scale in [("alpha", 360.0), ("delta", 180.0), ("ra", 360.0), ("dec", 180.0)]:
        if col in out.columns:
            rad = 2 * np.pi * out[col].astype(float) / scale
            out[f"{col}_sin"] = np.sin(rad)
            out[f"{col}_cos"] = np.cos(rad)
    out = out.replace([np.inf, -np.inf], np.nan)
    return out


def build_features(train: pd.DataFrame, test: pd.DataFrame, target_col: str):
    from sklearn.compose import ColumnTransformer
    from sklearn.impute import SimpleImputer
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import OneHotEncoder, StandardScaler

    train_fe = add_stellar_features(train)
    test_fe = add_stellar_features(test)
    drop_cols = [target_col]
    if "id" in train_fe.columns:
        drop_cols.append("id")
    X = train_fe.drop(columns=[c for c in drop_cols if c in train_fe.columns]).copy()
    Xt = test_fe.drop(columns=["id"], errors="ignore").copy()
    common = [c for c in X.columns if c in Xt.columns]
    X = X[common]
    Xt = Xt[common]

    numeric_cols = X.select_dtypes(include=[np.number, "bool"]).columns.tolist()
    categorical_cols = [c for c in X.columns if c not in numeric_cols]

    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler(with_mean=False)),
    ])
    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=5)),
    ])
    pre = ColumnTransformer([
        ("num", numeric_pipe, numeric_cols),
        ("cat", categorical_pipe, categorical_cols),
    ], remainder="drop", sparse_threshold=0.3)
    return X, Xt, pre, numeric_cols, categorical_cols


def candidate_estimators():
    estimators = []
    try:
        from sklearn.ensemble import ExtraTreesClassifier
        estimators.append(("extra_trees_420", ExtraTreesClassifier(
            n_estimators=420,
            max_features="sqrt",
            min_samples_leaf=1,
            class_weight="balanced_subsample",
            random_state=SEED + 13,
            n_jobs=-1,
        )))
    except Exception as e:
        print("ExtraTrees unavailable:", e)
    try:
        from sklearn.ensemble import RandomForestClassifier
        estimators.append(("random_forest_320", RandomForestClassifier(
            n_estimators=320,
            max_features="sqrt",
            min_samples_leaf=1,
            class_weight="balanced_subsample",
            random_state=SEED + 17,
            n_jobs=-1,
        )))
    except Exception as e:
        print("RandomForest unavailable:", e)
    try:
        from sklearn.linear_model import LogisticRegression
        estimators.append(("logreg_balanced", LogisticRegression(
            max_iter=1800,
            class_weight="balanced",
            C=1.8,
            n_jobs=-1,
            random_state=SEED + 19,
        )))
    except Exception as e:
        print("LogisticRegression unavailable:", e)

    if USE_OPTIONAL_MODELS:
        try:
            from sklearn.ensemble import HistGradientBoostingClassifier
            estimators.append(("hgb_tuned", HistGradientBoostingClassifier(
                learning_rate=0.045,
                max_iter=360,
                l2_regularization=0.02,
                max_leaf_nodes=43,
                random_state=SEED,
            )))
        except Exception:
            pass
        try:
            from lightgbm import LGBMClassifier
            estimators.append(("lightgbm_tuned", LGBMClassifier(
                objective="multiclass",
                n_estimators=900,
                learning_rate=0.032,
                num_leaves=79,
                min_child_samples=18,
                subsample=0.93,
                colsample_bytree=0.93,
                reg_lambda=0.02,
                random_state=SEED + 21,
                n_jobs=-1,
                verbose=-1,
            )))
        except Exception:
            pass
        try:
            from xgboost import XGBClassifier
            estimators.append(("xgboost_tuned", XGBClassifier(
                n_estimators=700,
                learning_rate=0.032,
                max_depth=7,
                min_child_weight=2,
                subsample=0.93,
                colsample_bytree=0.93,
                objective="multi:softprob",
                eval_metric="mlogloss",
                random_state=SEED + 31,
                n_jobs=-1,
                tree_method="hist",
            )))
        except Exception:
            pass
        try:
            from catboost import CatBoostClassifier
            estimators.append(("catboost_tuned", CatBoostClassifier(
                iterations=900,
                learning_rate=0.035,
                depth=7,
                loss_function="MultiClass",
                random_seed=SEED + 41,
                verbose=False,
                allow_writing_files=False,
            )))
        except Exception:
            pass
    return estimators


def tune_class_bias(oof_proba: np.ndarray, y: np.ndarray) -> np.ndarray:
    from sklearn.metrics import balanced_accuracy_score
    eps = 1e-12
    logits = np.log(np.clip(oof_proba, eps, 1.0))
    bias = np.zeros(logits.shape[1], dtype=float)
    best = balanced_accuracy_score(y, (logits + bias).argmax(axis=1))
    grid = np.linspace(-0.18, 0.18, 25)
    for _ in range(4):
        improved = False
        for c in range(logits.shape[1]):
            local_best = best
            local_bias = bias[c]
            for delta in grid:
                trial = bias.copy()
                trial[c] = delta
                score = balanced_accuracy_score(y, (logits + trial).argmax(axis=1))
                if score > local_best + 1e-9:
                    local_best = score
                    local_bias = delta
            if local_best > best + 1e-9:
                bias[c] = local_bias
                best = local_best
                improved = True
        if not improved:
            break
    print("OOF tuned class-bias balanced_accuracy:", round(best, 6), "bias:", bias)
    return bias


def run_model_fallback() -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame | None]:
    if train_path is None or test_path is None:
        raise RuntimeError("Model fallback requires real train.csv and test.csv. No toy predictions generated.")

    from sklearn.metrics import balanced_accuracy_score
    from sklearn.model_selection import StratifiedKFold
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import LabelEncoder

    train = pd.read_csv(train_path)
    test = pd.read_csv(test_path)
    target_col = infer_target_column(train)
    X, Xt, pre, numeric_cols, categorical_cols = build_features(train, test, target_col)
    y_raw = train[target_col].map(canonicalize_label)
    bad = sorted(set(y_raw.unique()) - set(CANONICAL_CLASSES))
    if bad:
        raise ValueError(f"Training target has illegal labels: {bad}")

    le = LabelEncoder()
    le.fit(CANONICAL_CLASSES)
    y = le.transform(y_raw)

    ests = candidate_estimators()
    if not ests:
        raise RuntimeError("No usable sklearn estimators available for actual model fallback.")

    n_splits = max(2, min(N_FOLDS, int(pd.Series(y).value_counts().min())))
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    test_proba_sum = np.zeros((len(test), len(CANONICAL_CLASSES)), dtype=np.float64)
    oof_blend = np.zeros((len(train), len(CANONICAL_CLASSES)), dtype=np.float64)
    oof_rows = []

    print("Model fallback target:", target_col)
    print("Feature count:", X.shape[1], "numeric:", len(numeric_cols), "categorical:", len(categorical_cols))
    print("Estimators:", [n for n, _ in ests])
    print("CV folds:", n_splits)

    model_weights = []
    model_test_probas = []
    model_oofs = []
    for model_name, estimator in ests:
        oof = np.zeros((len(train), len(CANONICAL_CLASSES)), dtype=np.float64)
        test_model_sum = np.zeros((len(test), len(CANONICAL_CLASSES)), dtype=np.float64)
        for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), start=1):
            pipe = Pipeline([("pre", pre), ("model", estimator)])
            pipe.fit(X.iloc[tr_idx], y[tr_idx])
            va_proba = pipe.predict_proba(X.iloc[va_idx])
            te_proba = pipe.predict_proba(Xt)
            classes = getattr(pipe.named_steps["model"], "classes_", np.arange(len(CANONICAL_CLASSES)))
            aligned_va = np.zeros((len(va_idx), len(CANONICAL_CLASSES)))
            aligned_te = np.zeros((len(test), len(CANONICAL_CLASSES)))
            for pos, cls in enumerate(classes):
                aligned_va[:, int(cls)] = va_proba[:, pos]
                aligned_te[:, int(cls)] = te_proba[:, pos]
            oof[va_idx] = aligned_va
            test_model_sum += aligned_te / n_splits
            fold_pred = oof[va_idx].argmax(axis=1)
            fold_bal = balanced_accuracy_score(y[va_idx], fold_pred)
            oof_rows.append({"model": model_name, "fold": fold, "balanced_accuracy": fold_bal})
            print(f"{model_name} fold {fold}: balanced_accuracy={fold_bal:.6f}")
        full_bal = balanced_accuracy_score(y, oof.argmax(axis=1))
        oof_rows.append({"model": model_name, "fold": "OOF", "balanced_accuracy": full_bal})
        print(f"{model_name} OOF: balanced_accuracy={full_bal:.6f}")
        weight = max(0.001, full_bal - 0.80) ** 4
        model_weights.append(weight)
        model_test_probas.append(test_model_sum)
        model_oofs.append(oof)

    weights = np.asarray(model_weights, dtype=float)
    weights = weights / max(weights.sum(), 1e-12)
    for w, oof, te in zip(weights, model_oofs, model_test_probas):
        oof_blend += w * oof
        test_proba_sum += w * te

    base_oof = balanced_accuracy_score(y, oof_blend.argmax(axis=1))
    bias = tune_class_bias(oof_blend, y)
    eps = 1e-12
    pred_i = (np.log(np.clip(test_proba_sum, eps, 1.0)) + bias).argmax(axis=1)
    sub = sample[["id"]].copy()
    sub["class"] = le.inverse_transform(pred_i)
    sub = validate_id_class(sub, sample, name="model_fallback")

    oof_report = pd.DataFrame(oof_rows)
    oof_report.loc[len(oof_report)] = {"model": "blend", "fold": "OOF", "balanced_accuracy": base_oof}
    oof_report.to_csv(OOF_CSV, index=False)

    proba_debug = sample[["id"]].copy()
    for i, cls in enumerate(CANONICAL_CLASSES):
        proba_debug[f"model_proba_{cls}"] = test_proba_sum[:, i]
    proba_debug["model_pred"] = sub["class"].values

    write_report("Model fallback", [f"OOF report saved: `{OOF_CSV}`", f"Submission rows: `{len(sub)}`"])
    return sub, oof_report, proba_debug


## 6. Build guardian micro-flip candidates 🛡️


In [9]:
def candidate_stats(name: str, sub: pd.DataFrame, anchor: pd.DataFrame | None = None, conf: pd.DataFrame | None = None) -> dict:
    row = {"candidate": name, "rows": len(sub), "hash": dataframe_hash(sub)}
    for c in CANONICAL_CLASSES:
        row[c] = int((sub["class"] == c).sum())
    if anchor is not None:
        mask = sub["class"].values != anchor["class"].values
        row["changed_vs_anchor"] = int(mask.sum())
        row["changed_rate_vs_anchor"] = float(mask.mean())
        if conf is not None and len(conf) == len(sub):
            row["mean_changed_vote_share"] = float(conf.loc[mask, "vote_share"].mean()) if mask.any() else np.nan
            row["mean_changed_vote_margin"] = float(conf.loc[mask, "vote_margin"].mean()) if mask.any() else np.nan
            row["min_changed_agree_count"] = float(conf.loc[mask, "agree_count"].min()) if mask.any() else np.nan
    return row


def guardian_override(anchor_sub: pd.DataFrame, stack_pred: pd.Series, conf: pd.DataFrame, *, min_share: float, min_margin: float, min_agree: int, max_flip_rows: int | None = None) -> tuple[pd.DataFrame, pd.DataFrame]:
    out = anchor_sub.copy().reset_index(drop=True)
    stack_pred = stack_pred.reset_index(drop=True)
    anchor_pred = out["class"].reset_index(drop=True)
    mask = (
        (stack_pred != anchor_pred)
        & (conf["vote_share"].to_numpy() >= min_share)
        & (conf["vote_margin"].to_numpy() >= min_margin)
        & (conf["agree_count"].to_numpy() >= min_agree)
    )
    idx = np.where(mask)[0]
    if max_flip_rows is not None and len(idx) > max_flip_rows:
        # Keep only the highest-confidence flips.
        order = np.lexsort((-conf.loc[idx, "vote_margin"].to_numpy(), -conf.loc[idx, "vote_share"].to_numpy()))
        keep_idx = idx[order[:max_flip_rows]]
        new_mask = np.zeros(len(mask), dtype=bool)
        new_mask[keep_idx] = True
        mask = new_mask
    out.loc[mask, "class"] = stack_pred.loc[mask].values
    flip_log = pd.DataFrame({
        "id": sample["id"].values,
        "anchor_class": anchor_pred.values,
        "stack_class": stack_pred.values,
        "final_class": out["class"].values,
        "flipped": mask,
        "vote_share": conf["vote_share"].values,
        "vote_margin": conf["vote_margin"].values,
        "agree_count": conf["agree_count"].values,
    })
    return validate_id_class(out, sample, name="guardian_candidate"), flip_log


def build_stack_candidates() -> tuple[dict[str, pd.DataFrame], dict[str, pd.DataFrame], pd.DataFrame]:
    candidates: dict[str, pd.DataFrame] = {}
    diagnostics: dict[str, pd.DataFrame] = {}
    manifest_rows = []
    if not ranked_keys:
        return candidates, diagnostics, pd.DataFrame()

    anchor = validate_id_class(source_frames[anchor_key or ranked_keys[0]], sample, name="anchor")
    candidates["anchor_lock"] = anchor
    manifest_rows.append(candidate_stats("anchor_lock", anchor, anchor=anchor))

    max_flip_rows = max(1, int(math.ceil(len(sample) * MAX_FLIP_RATE)))
    top_grid = sorted(set([3, 5, 7, 9, 11, 15, min(25, len(ranked_keys)), len(ranked_keys)]))
    top_grid = [n for n in top_grid if n >= 1 and n <= len(ranked_keys)]
    temp_grid = [2500.0, 5000.0, SOURCE_TEMP_DEFAULT, 9000.0]

    for n in top_grid:
        keys = ranked_keys[:n]
        # Majority stacks.
        if n >= 3:
            maj_pred = majority_vote_from_sources(keys)
            name = f"majority_top{n}"
            sub = make_submission(maj_pred, sample)
            candidates[name] = sub
            manifest_rows.append(candidate_stats(name, sub, anchor=anchor))
        # Weighted stacks and guardian variants.
        for temp in temp_grid:
            w_pred, conf = weighted_vote_from_sources(keys, temperature=temp)
            wname = f"weighted_top{n}_t{int(temp)}"
            wsub = make_submission(w_pred, sample)
            candidates[wname] = wsub
            diagnostics[f"conf_{wname}"] = conf
            manifest_rows.append(candidate_stats(wname, wsub, anchor=anchor, conf=conf))

            for share in [0.80, 0.84, 0.88, 0.92, 0.96]:
                if share < STRICT_MARGIN:
                    continue
                for agree in sorted(set([MIN_AGREE, max(MIN_AGREE, 4), max(MIN_AGREE, min(6, n))])):
                    if agree > n:
                        continue
                    min_margin = max(0.12, share - 0.55)
                    gsub, flog = guardian_override(anchor, w_pred, conf, min_share=share, min_margin=min_margin, min_agree=agree, max_flip_rows=max_flip_rows)
                    gname = f"guardian_top{n}_t{int(temp)}_s{int(share*100)}_a{agree}"
                    candidates[gname] = gsub
                    diagnostics[f"flips_{gname}"] = flog[flog["flipped"]].copy()
                    manifest_rows.append(candidate_stats(gname, gsub, anchor=anchor, conf=conf))

    manifest = pd.DataFrame(manifest_rows).drop_duplicates("hash", keep="first")
    if len(manifest):
        # Safety score: prefers nonzero, very small, high-confidence changes vs anchor.
        max_rows = max(1, len(sample))
        changed = manifest.get("changed_vs_anchor", pd.Series(np.zeros(len(manifest)))).fillna(0)
        share = manifest.get("mean_changed_vote_share", pd.Series(np.zeros(len(manifest)))).fillna(0)
        margin = manifest.get("mean_changed_vote_margin", pd.Series(np.zeros(len(manifest)))).fillna(0)
        agree = manifest.get("min_changed_agree_count", pd.Series(np.zeros(len(manifest)))).fillna(0)
        risk = changed / max_rows
        manifest["guardian_safety_score"] = (share * 4.0 + margin * 2.5 + np.log1p(agree)) - (risk * 100.0)
        manifest["is_microflip"] = (changed > 0) & (changed <= max_flip_rows)
        manifest = manifest.sort_values(["is_microflip", "guardian_safety_score", "changed_vs_anchor"], ascending=[False, False, True]).reset_index(drop=True)
    return candidates, diagnostics, manifest

stack_candidates, stack_diag, stack_manifest = build_stack_candidates()
print("Stack candidate count:", len(stack_candidates))
display(stack_manifest.head(30))


Stack candidate count: 393


,candidate,rows,hash,GALAXY,STAR,QSO,changed_vs_anchor,changed_rate_vs_anchor,mean_changed_vote_share,mean_changed_vote_margin,min_changed_agree_count,guardian_safety_score,is_microflip
0,weighted_top56_t2500,247435,d2059085935d5285,157065,38939,51431,371,0.001499,0.596264,0.193073,25.0,5.975896,True
1,weighted_top56_t5000,247435,f083b04a3e67b1f6,156947,39024,51464,82,0.000331,0.559820,0.119792,28.0,5.872914,True
2,weighted_top56_t6500,247435,0d603233da498e48,156907,39057,51471,27,0.000109,0.521110,0.042219,39.0,5.867954,True
3,weighted_top25_t2500,247435,e447711d66612fc2,157021,38969,51445,243,0.000982,0.591048,0.182535,14.0,5.430371,True
4,weighted_top25_t5000,247435,2f4f319d8168a1c4,156935,39037,51463,63,0.000255,0.561940,0.123880,17.0,5.422369,True
5,weighted_top25_t6500,247435,80b457753390f2e2,156904,39060,51471,24,0.000097,0.513424,0.026848,21.0,5.202158,True
6,weighted_top15_t2500,247435,c16ccdc5c210b333,156960,39016,51459,97,0.000392,0.580691,0.161382,8.0,4.884243,True
7,weighted_top15_t5000,247435,58b1c7face7f5d2c,156911,39055,51469,33,0.000133,0.535817,0.071635,10.0,4.706915,True
8,weighted_top9_t2500,247435,9140f757cd0c3946,156939,39034,51462,69,0.000279,0.562962,0.125924,6.0,4.484684,True
9,weighted_top11_t2500,247435,60ecc00dd071e2a8,156947,39028,51460,85,0.000344,0.552881,0.105762,6.0,4.387488,True


## 7. Optional model-assist candidate injection 🔧


In [10]:
model_sub = None
model_oof = None
model_debug = None

source_missing = not bool(stack_candidates)
guardian_like_mode = MODE in {"auto", "guardian_97122", "anchor_lock", "best_lock", "weighted_stack", "majority_stack"}
if source_missing and REQUIRE_SOURCE and guardian_like_mode and not ALLOW_MODEL_ONLY:
    raise RuntimeError(
        "No valid source/anchor submissions were discovered, so guardian mode stopped before writing a likely low-score fallback. "
        "Attach your 0.97122 submission CSV as a Kaggle Dataset input. Direct files named `submission.csv` are accepted in v2. "
        "Or set KAGGLE_AR_MODE=model_fallback / KAGGLE_AR_ALLOW_MODEL_ONLY=1 if you intentionally want model-only output."
    )

need_model = (MODE == "model_fallback") or MODEL_ASSIST or (source_missing and ALLOW_MODEL_ONLY and guardian_like_mode)

if need_model:
    model_sub, model_oof, model_debug = run_model_fallback()
    stack_candidates["model_fallback"] = model_sub
    base_anchor = stack_candidates.get("anchor_lock", model_sub)
    extra = pd.DataFrame([candidate_stats("model_fallback", model_sub, anchor=base_anchor)])
    stack_manifest = pd.concat([stack_manifest, extra], ignore_index=True) if len(stack_manifest) else extra
    if model_debug is not None:
        model_debug.to_csv(AR_DIR / "model_test_proba_debug.csv", index=False)

# Optional low-weight model-assist guardian: only if source consensus exists and model agrees with the proposed flip.
if MODEL_ASSIST and model_sub is not None and "anchor_lock" in stack_candidates and ranked_keys:
    anchor = stack_candidates["anchor_lock"]
    keys = ranked_keys[:min(15, len(ranked_keys))]
    w_pred, conf = weighted_vote_from_sources(keys, temperature=SOURCE_TEMP_DEFAULT)
    model_agrees = model_sub["class"].reset_index(drop=True).eq(w_pred.reset_index(drop=True))
    conf2 = conf.copy()
    conf2["vote_share"] = np.where(model_agrees, np.minimum(1.0, conf2["vote_share"] + 0.03), conf2["vote_share"])
    conf2["vote_margin"] = np.where(model_agrees, np.minimum(1.0, conf2["vote_margin"] + 0.03), conf2["vote_margin"])
    conf2["agree_count"] = np.where(model_agrees, conf2["agree_count"] + 1, conf2["agree_count"])
    assist_sub, assist_flips = guardian_override(anchor, w_pred, conf2, min_share=max(STRICT_MARGIN, 0.88), min_margin=0.33, min_agree=max(MIN_AGREE + 1, 4), max_flip_rows=max(1, int(math.ceil(len(sample) * MAX_FLIP_RATE))))
    stack_candidates["guardian_model_assist"] = assist_sub
    stack_diag["flips_guardian_model_assist"] = assist_flips[assist_flips["flipped"]].copy()
    stack_manifest = pd.concat([stack_manifest, pd.DataFrame([candidate_stats("guardian_model_assist", assist_sub, anchor=anchor, conf=conf2)])], ignore_index=True)

print("Candidate count after model stage:", len(stack_candidates))
if len(stack_manifest):
    display(stack_manifest.tail(20))


Candidate count after model stage: 393


,candidate,rows,hash,GALAXY,STAR,QSO,changed_vs_anchor,changed_rate_vs_anchor,mean_changed_vote_share,mean_changed_vote_margin,min_changed_agree_count,guardian_safety_score,is_microflip
0,weighted_top56_t2500,247435,d2059085935d5285,157065,38939,51431,371,0.001499,0.596264,0.193073,25.0,5.975896,True
1,weighted_top56_t5000,247435,f083b04a3e67b1f6,156947,39024,51464,82,0.000331,0.559820,0.119792,28.0,5.872914,True
2,weighted_top56_t6500,247435,0d603233da498e48,156907,39057,51471,27,0.000109,0.521110,0.042219,39.0,5.867954,True
3,weighted_top25_t2500,247435,e447711d66612fc2,157021,38969,51445,243,0.000982,0.591048,0.182535,14.0,5.430371,True
4,weighted_top25_t5000,247435,2f4f319d8168a1c4,156935,39037,51463,63,0.000255,0.561940,0.123880,17.0,5.422369,True
5,weighted_top25_t6500,247435,80b457753390f2e2,156904,39060,51471,24,0.000097,0.513424,0.026848,21.0,5.202158,True
6,weighted_top15_t2500,247435,c16ccdc5c210b333,156960,39016,51459,97,0.000392,0.580691,0.161382,8.0,4.884243,True
7,weighted_top15_t5000,247435,58b1c7face7f5d2c,156911,39055,51469,33,0.000133,0.535817,0.071635,10.0,4.706915,True
8,weighted_top9_t2500,247435,9140f757cd0c3946,156939,39034,51462,69,0.000279,0.562962,0.125924,6.0,4.484684,True
9,weighted_top11_t2500,247435,60ecc00dd071e2a8,156947,39028,51460,85,0.000344,0.552881,0.105762,6.0,4.387488,True


## 8B. Borg/Locutus collective voter layer 🧬

Additive only: keeps the 0.97122 anchor lock intact, creates Borg majority/weighted candidates from selected ranked sources, then creates ultra-strict guardian micro-flip candidates. No toy predictions.

In [ ]:
# ============================================================
# 8B. Borg/Locutus collective voter layer — additive recovery lock
# ============================================================
# Design rules:
# - Preserve anchor_lock as the safe baseline.
# - Use ranked source positions [3, 15, 16, 17, 52] when available.
# - Skip unavailable voter positions safely and fill from top-ranked sources.
# - Add strict Borg candidates to stack_candidates; final selector remains the gatekeeper.
# - Never create random/toy predictions.

BORG_ENABLE = os.getenv("KAGGLE_BORG_ENABLE", "1").strip().lower() not in {"0", "false", "no", "off"}
BORG_VOTERS_RAW = os.getenv("KAGGLE_BORG_VOTERS", "3,15,16,17,52").strip()
BORG_MIN_AGREE = int(os.getenv("KAGGLE_BORG_MIN_AGREE", "5"))
BORG_MAX_FLIP_RATE = float(os.getenv("KAGGLE_BORG_MAX_FLIP_RATE", str(MAX_FLIP_RATE)))
BORG_REPORT_CSV = AR_DIR / "borg_voter_report.csv"


def parse_int_list(raw: str) -> list[int]:
    vals: list[int] = []
    for part in re.split(r"[,;\s]+", raw.strip()):
        if not part:
            continue
        try:
            vals.append(int(part))
        except ValueError:
            continue
    return vals


def resolve_borg_voter_keys(ranked: list[str], raw_positions: str, *, min_voters: int = 5) -> tuple[list[str], pd.DataFrame]:
    """Resolve 1-based ranked source positions into real source keys.

    Example: raw_positions="3,15,16,17,52" selects ranked_keys[2], ranked_keys[14], ...
    Missing positions are logged, not fatal. If fewer than min_voters exist, fill from top-ranked keys.
    """
    positions = parse_int_list(raw_positions)
    selected: list[str] = []
    rows = []
    for pos in positions:
        if 1 <= pos <= len(ranked):
            key = ranked[pos - 1]
            if key not in selected:
                selected.append(key)
            rows.append({"requested_position": pos, "resolved": True, "key": key})
        else:
            rows.append({"requested_position": pos, "resolved": False, "key": ""})

    target_n = min(max(1, min_voters), len(ranked)) if ranked else 0
    for key in ranked:
        if len(selected) >= target_n:
            break
        if key not in selected:
            selected.append(key)
            rows.append({"requested_position": "fill", "resolved": True, "key": key})

    return selected, pd.DataFrame(rows)


def borg_vote_conf(keys: list[str], *, temperature: float = SOURCE_TEMP_DEFAULT) -> tuple[pd.Series, pd.DataFrame]:
    """Weighted Borg vote plus raw count diagnostics."""
    pred, conf = weighted_vote_from_sources(keys, temperature=temperature)
    mat = source_matrix(keys)
    raw_counts = np.zeros((mat.shape[1], len(CANONICAL_CLASSES)), dtype=np.int16)
    for row in mat:
        raw_counts[np.arange(mat.shape[1]), row] += 1
    conf = conf.copy()
    conf["borg_count_GALAXY"] = raw_counts[:, CLASS_TO_INT["GALAXY"]]
    conf["borg_count_STAR"] = raw_counts[:, CLASS_TO_INT["STAR"]]
    conf["borg_count_QSO"] = raw_counts[:, CLASS_TO_INT["QSO"]]
    conf["borg_raw_agree_count"] = raw_counts.max(axis=1)
    conf["borg_raw_vote_share"] = raw_counts.max(axis=1) / max(1, len(keys))
    return pred, conf


def build_borg_collective_candidates() -> None:
    global stack_manifest

    if not BORG_ENABLE:
        print("Borg layer disabled by KAGGLE_BORG_ENABLE=0")
        return
    if not ranked_keys:
        print("Borg layer skipped: no ranked source submissions.")
        return
    if "anchor_lock" not in stack_candidates:
        print("Borg layer skipped: anchor_lock missing.")
        return

    anchor = validate_id_class(stack_candidates["anchor_lock"], sample, name="borg_anchor")
    borg_keys, voter_report = resolve_borg_voter_keys(ranked_keys, BORG_VOTERS_RAW, min_voters=min(5, len(ranked_keys)))
    if not borg_keys:
        print("Borg layer skipped: no resolved voters.")
        return

    meta_idx = source_meta.set_index("key") if len(source_meta) else pd.DataFrame()
    enriched_rows = []
    for order, key in enumerate(borg_keys, start=1):
        row = {"borg_order": order, "key": key}
        if len(meta_idx) and key in meta_idx.index:
            for col in ["filename", "filename_score", "agreement_centrality", "duplicate_count", "hash", "relpath"]:
                row[col] = meta_idx.loc[key].get(col, "")
        enriched_rows.append(row)
    voter_report = voter_report.merge(pd.DataFrame(enriched_rows), on="key", how="left") if len(voter_report) else pd.DataFrame(enriched_rows)
    voter_report.to_csv(BORG_REPORT_CSV, index=False)

    print("BORG voters:", borg_keys)
    display(voter_report)

    manifest_add = []
    max_flip_rows = max(1, int(math.ceil(len(sample) * BORG_MAX_FLIP_RATE)))
    min_agree = min(max(1, BORG_MIN_AGREE), len(borg_keys))

    # Majority candidate, useful for diagnostics and manual candidate testing.
    if len(borg_keys) >= 3:
        maj_pred = majority_vote_from_sources(borg_keys)
        maj_name = f"borg_majority_v{len(borg_keys)}"
        maj_sub = make_submission(maj_pred, sample)
        stack_candidates[maj_name] = maj_sub
        manifest_add.append(candidate_stats(maj_name, maj_sub, anchor=anchor))

    # Weighted candidate and strict guardian micro-flip candidates.
    for temp in sorted(set([2500.0, 5000.0, SOURCE_TEMP_DEFAULT, 9000.0, 12000.0])):
        borg_pred, borg_conf = borg_vote_conf(borg_keys, temperature=temp)
        w_name = f"borg_weighted_v{len(borg_keys)}_t{int(temp)}"
        w_sub = make_submission(borg_pred, sample)
        stack_candidates[w_name] = w_sub
        stack_diag[f"conf_{w_name}"] = borg_conf
        manifest_add.append(candidate_stats(w_name, w_sub, anchor=anchor, conf=borg_conf))

        # Strict Borg guard: raw voter agreement + weighted confidence. This prevents broad drift.
        for share in [0.88, 0.92, 0.96, 0.995]:
            g_conf = borg_conf.copy()
            g_conf["agree_count"] = np.maximum(g_conf["agree_count"].to_numpy(), g_conf["borg_raw_agree_count"].to_numpy())
            g_conf["vote_share"] = np.minimum(g_conf["vote_share"].to_numpy(), g_conf["borg_raw_vote_share"].to_numpy())
            min_margin = max(0.25, share - 0.58)
            g_sub, g_log = guardian_override(
                anchor,
                borg_pred,
                g_conf,
                min_share=share,
                min_margin=min_margin,
                min_agree=min_agree,
                max_flip_rows=max_flip_rows,
            )
            g_name = f"guardian_borg_v{len(borg_keys)}_t{int(temp)}_s{int(share*1000)}_a{min_agree}"
            stack_candidates[g_name] = g_sub
            stack_diag[f"flips_{g_name}"] = g_log[g_log["flipped"]].copy()
            manifest_add.append(candidate_stats(g_name, g_sub, anchor=anchor, conf=g_conf))

        # Ultra strict unanimous lock: only flip where all Borg voters agree against anchor.
        u_conf = borg_conf.copy()
        u_conf["agree_count"] = u_conf["borg_raw_agree_count"].to_numpy()
        u_conf["vote_share"] = u_conf["borg_raw_vote_share"].to_numpy()
        u_sub, u_log = guardian_override(
            anchor,
            borg_pred,
            u_conf,
            min_share=1.0,
            min_margin=0.50,
            min_agree=len(borg_keys),
            max_flip_rows=max_flip_rows,
        )
        u_name = f"guardian_borg_unanimous_v{len(borg_keys)}_t{int(temp)}"
        stack_candidates[u_name] = u_sub
        stack_diag[f"flips_{u_name}"] = u_log[u_log["flipped"]].copy()
        manifest_add.append(candidate_stats(u_name, u_sub, anchor=anchor, conf=u_conf))

    if manifest_add:
        add_df = pd.DataFrame(manifest_add).drop_duplicates("hash", keep="first")
        max_rows = max(1, len(sample))
        changed = add_df.get("changed_vs_anchor", pd.Series(np.zeros(len(add_df)))).fillna(0)
        share = add_df.get("mean_changed_vote_share", pd.Series(np.zeros(len(add_df)))).fillna(0)
        margin = add_df.get("mean_changed_vote_margin", pd.Series(np.zeros(len(add_df)))).fillna(0)
        agree = add_df.get("min_changed_agree_count", pd.Series(np.zeros(len(add_df)))).fillna(0)
        risk = changed / max_rows
        add_df["guardian_safety_score"] = (share * 4.5 + margin * 3.0 + np.log1p(agree)) - (risk * 120.0)
        add_df["is_microflip"] = (changed > 0) & (changed <= max_flip_rows)
        stack_manifest = pd.concat([stack_manifest, add_df], ignore_index=True) if len(stack_manifest) else add_df
        stack_manifest = stack_manifest.drop_duplicates("hash", keep="first")
        stack_manifest = stack_manifest.sort_values(
            ["is_microflip", "guardian_safety_score", "changed_vs_anchor"],
            ascending=[False, False, True],
        ).reset_index(drop=True)

    write_report("Borg collective layer", [
        f"Enabled: `{BORG_ENABLE}`",
        f"Requested voters: `{BORG_VOTERS_RAW}`",
        f"Resolved voters: `{len(borg_keys)}`",
        f"Min agree: `{min_agree}`",
        f"Max flip rows: `{max_flip_rows}`",
        f"Voter report: `{BORG_REPORT_CSV}`",
    ])
    print("Borg candidate count added:", len(manifest_add))
    display(stack_manifest[stack_manifest["candidate"].astype(str).str.contains("borg", case=False, na=False)].head(40))


build_borg_collective_candidates()
print("Candidate count after Borg layer:", len(stack_candidates))


## 8. Select final candidate and write candidate pack ✅


In [11]:
def choose_final_candidate() -> tuple[str, pd.DataFrame]:
    if not stack_candidates:
        raise RuntimeError("No valid stack sources and no train/test fallback available. No toy predictions generated.")

    anchor = stack_candidates.get("anchor_lock")
    manifest = []
    for name, sub in stack_candidates.items():
        manifest.append(candidate_stats(name, sub, anchor=anchor if anchor is not None else sub))
    manifest = pd.DataFrame(manifest).drop_duplicates("hash", keep="first")

    if len(stack_manifest):
        keep_cols = [c for c in ["candidate", "guardian_safety_score", "is_microflip", "mean_changed_vote_share", "mean_changed_vote_margin", "min_changed_agree_count"] if c in stack_manifest.columns]
        if keep_cols:
            manifest = manifest.merge(stack_manifest[keep_cols].drop_duplicates("candidate"), on="candidate", how="left")

    for name, sub in stack_candidates.items():
        # Write every candidate for manual submit testing.
        safe_name = re.sub(r"[^A-Za-z0-9_.-]+", "_", name)[:160]
        sub.to_csv(CAND_DIR / f"{safe_name}.csv", index=False)

    mode = MODE
    selected = None
    if mode in {"anchor_lock", "best_lock"}:
        selected = "anchor_lock" if "anchor_lock" in stack_candidates else None
    elif mode == "weighted_stack":
        weighted_names = [n for n in stack_candidates if n.startswith("weighted_top")]
        selected = weighted_names[0] if weighted_names else None
    elif mode == "majority_stack":
        majority_names = [n for n in stack_candidates if n.startswith("majority_top")]
        selected = majority_names[0] if majority_names else None
    elif mode == "model_fallback":
        selected = "model_fallback" if "model_fallback" in stack_candidates else None
    elif mode in {"guardian_97122", "auto"}:
        max_flip_rows = max(1, int(math.ceil(len(sample) * MAX_FLIP_RATE)))
        mm = manifest.copy()
        if "changed_vs_anchor" not in mm.columns:
            mm["changed_vs_anchor"] = 0
        if "guardian_safety_score" not in mm.columns:
            mm["guardian_safety_score"] = 0.0
        micro = mm[(mm["candidate"].str.startswith("guardian_")) & (mm["changed_vs_anchor"] > 0) & (mm["changed_vs_anchor"] <= max_flip_rows)].copy()
        if len(micro):
            micro = micro.sort_values(["guardian_safety_score", "changed_vs_anchor"], ascending=[False, True])
            selected = str(micro.iloc[0]["candidate"])
        elif "anchor_lock" in stack_candidates:
            selected = "anchor_lock"
        elif "model_fallback" in stack_candidates:
            selected = "model_fallback"
    elif mode in {"borg", "borg_collective", "locutus", "locutus_collective"}:
        max_flip_rows = max(1, int(math.ceil(len(sample) * float(os.getenv("KAGGLE_BORG_MAX_FLIP_RATE", str(MAX_FLIP_RATE))))))
        mm = manifest.copy()
        if "changed_vs_anchor" not in mm.columns:
            mm["changed_vs_anchor"] = 0
        if "guardian_safety_score" not in mm.columns:
            mm["guardian_safety_score"] = 0.0
        borg_micro = mm[
            mm["candidate"].astype(str).str.startswith("guardian_borg_")
            & (mm["changed_vs_anchor"] > 0)
            & (mm["changed_vs_anchor"] <= max_flip_rows)
        ].copy()
        if len(borg_micro):
            borg_micro = borg_micro.sort_values(["guardian_safety_score", "changed_vs_anchor"], ascending=[False, True])
            selected = str(borg_micro.iloc[0]["candidate"])
        else:
            borg_weighted = [n for n in stack_candidates if n.startswith("borg_weighted_")]
            selected = borg_weighted[0] if borg_weighted else ("anchor_lock" if "anchor_lock" in stack_candidates else None)
    else:
        if mode in stack_candidates:
            selected = mode
        else:
            # Allow exact candidate names via environment.
            matches = [n for n in stack_candidates if n.lower() == mode]
            selected = matches[0] if matches else None

    if selected is None or selected not in stack_candidates:
        available = sorted(stack_candidates)[:50]
        raise ValueError(f"Could not select mode={MODE!r}. First available candidates: {available}")

    final = validate_id_class(stack_candidates[selected], sample, name="final")

    # Final manifest sorted for review.
    if len(manifest):
        if "changed_vs_anchor" not in manifest.columns:
            manifest["changed_vs_anchor"] = 0
        if "guardian_safety_score" not in manifest.columns:
            manifest["guardian_safety_score"] = 0.0
        manifest["selected"] = manifest["candidate"].eq(selected)
        manifest = manifest.sort_values(["selected", "guardian_safety_score", "changed_vs_anchor"], ascending=[False, False, True]).reset_index(drop=True)
        manifest.to_csv(CANDIDATES_CSV, index=False)

    # Debug output.
    debug = sample[["id"]].copy()
    for k in ["anchor_lock", selected, "model_fallback"]:
        if k in stack_candidates and k not in debug.columns:
            debug[k] = stack_candidates[k]["class"].values
    debug["final"] = final["class"].values
    if "anchor_lock" in stack_candidates:
        debug["changed_vs_anchor"] = debug["final"].values != stack_candidates["anchor_lock"]["class"].values
    debug.to_csv(DEBUG_CSV, index=False)

    # Flip log for selected candidate if available.
    flip_key = f"flips_{selected}"
    if flip_key in stack_diag:
        stack_diag[flip_key].to_csv(FLIPS_CSV, index=False)
    else:
        pd.DataFrame().to_csv(FLIPS_CSV, index=False)

    write_report("Final selection", [
        f"Selected candidate: `{selected}`",
        f"Output: `{OUT}`",
        f"Candidate manifest: `{CANDIDATES_CSV}`",
        f"Candidate pack: `{CAND_DIR}`",
        f"Debug predictions: `{DEBUG_CSV}`",
        f"Flip rows: `{FLIPS_CSV}`",
    ])
    return selected, final

selected_name, final_sub = choose_final_candidate()
final_sub.to_csv(OUT, index=False)

print("SELECTED:", selected_name)
print("WROTE:", OUT, OUT.stat().st_size, "bytes")
print("REPORT:", REPORT_MD)
print("MANIFEST:", CANDIDATES_CSV)
print("CANDIDATE PACK:", CAND_DIR)
print("DEBUG:", DEBUG_CSV)
if CANDIDATES_CSV.exists():
    display(pd.read_csv(CANDIDATES_CSV).head(40))
display(final_sub.head())
print(final_sub["class"].value_counts())


SELECTED: anchor_lock
WROTE: /kaggle/working/submission.csv 3231519 bytes
REPORT: /kaggle/working/kaggle_autoresearch_report.md
MANIFEST: /kaggle/working/autoresearch/candidate_manifest.csv
CANDIDATE PACK: /kaggle/working/autoresearch/candidates
DEBUG: /kaggle/working/autoresearch/debug_predictions.csv


,candidate,rows,hash,GALAXY,STAR,QSO,changed_vs_anchor,changed_rate_vs_anchor,guardian_safety_score,is_microflip,mean_changed_vote_share,mean_changed_vote_margin,min_changed_agree_count,selected
0,anchor_lock,247435,4fa2346b2a213bd4,156886,39067,51482,0,0.000000,0.000000,False,NaN,NaN,NaN,True
1,weighted_top56_t2500,247435,d2059085935d5285,157065,38939,51431,371,0.001499,5.975896,True,0.596264,0.193073,25.0,False
2,weighted_top56_t5000,247435,f083b04a3e67b1f6,156947,39024,51464,82,0.000331,5.872914,True,0.559820,0.119792,28.0,False
3,weighted_top56_t6500,247435,0d603233da498e48,156907,39057,51471,27,0.000109,5.867954,True,0.521110,0.042219,39.0,False
4,weighted_top25_t2500,247435,e447711d66612fc2,157021,38969,51445,243,0.000982,5.430371,True,0.591048,0.182535,14.0,False
5,weighted_top25_t5000,247435,2f4f319d8168a1c4,156935,39037,51463,63,0.000255,5.422369,True,0.561940,0.123880,17.0,False
6,weighted_top25_t6500,247435,80b457753390f2e2,156904,39060,51471,24,0.000097,5.202158,True,0.513424,0.026848,21.0,False
7,weighted_top15_t2500,247435,c16ccdc5c210b333,156960,39016,51459,97,0.000392,4.884243,True,0.580691,0.161382,8.0,False
8,weighted_top15_t5000,247435,58b1c7face7f5d2c,156911,39055,51469,33,0.000133,4.706915,True,0.535817,0.071635,10.0,False
9,weighted_top9_t2500,247435,9140f757cd0c3946,156939,39034,51462,69,0.000279,4.484684,True,0.562962,0.125924,6.0,False


,id,class
0,577347,GALAXY
1,577348,GALAXY
2,577349,GALAXY
3,577350,STAR
4,577351,GALAXY


class
GALAXY    156886
QSO        51482
STAR       39067
Name: count, dtype: int64


## 9. Final validation gate 🧷


In [12]:
check = pd.read_csv(OUT)
check = validate_id_class(check, sample, name="submission.csv")
assert check["id"].equals(sample["id"]), "submission id order changed"
assert set(check["class"].unique()).issubset(CANONICAL_CLASSES), "illegal final class labels"

print("submission.csv ready ✅")
print(check.shape)
print(check.head())
print("Candidate files written:", len(list(CAND_DIR.glob("*.csv"))))

print("Borg voter report:", BORG_REPORT_CSV if "BORG_REPORT_CSV" in globals() else "<not enabled>")
print("Selected candidate:", selected_name if "selected_name" in globals() else "<unknown>")


submission.csv ready ✅
(247435, 2)
       id   class
0  577347  GALAXY
1  577348  GALAXY
2  577349  GALAXY
3  577350    STAR
4  577351  GALAXY
Candidate files written: 393
